# 29 孟加拉国 DIA 冻结谱图模型输入转换（V2）

严格复用 Notebook 20 的四类部署输入契约：binned MS/MS、DreaMS、Set Transformer 和 Notebook 18 Transformer。随后提取 DreaMS 嵌入并冻结。

必须使用 `env_dreams` 内核和 CUDA。

In [ ]:
from pathlib import Path
import sys,json
ROOT=Path.cwd().resolve()
while ROOT!=ROOT.parent and not (ROOT/'src'/'pbt_msms').exists(): ROOT=ROOT.parent
assert (ROOT/'src'/'pbt_msms').exists(), '找不到项目根目录'
if str(ROOT/'src') not in sys.path: sys.path.insert(0,str(ROOT/'src'))
import torch
print('ROOT =',ROOT); print('Python =',sys.executable); print('CUDA =',torch.cuda.is_available())
assert torch.cuda.is_available(), '请切换到 env_dreams CUDA 内核'


In [ ]:
from pbt_msms.bangladesh_dia_external import run_prepare,paths
prepare=run_prepare(ROOT)
print(json.dumps(prepare,ensure_ascii=False,indent=2))
assert prepare['status']=='READY_FOR_DREAMS_EMBEDDING'


In [ ]:
# DreaMS 嵌入可能耗时；显存不足时将 batch_size 改为 16。
from pbt_msms.bangladesh_dia_external import run_dreams_embedding
embedding=run_dreams_embedding(ROOT,batch_size=32)
print(json.dumps(embedding,ensure_ascii=False,indent=2))
assert embedding['status']=='READY_FOR_FROZEN_MODEL_INFERENCE'


In [ ]:
import pandas as pd,numpy as np
p=paths(ROOT); index=pd.read_csv(p['index'])
emb=np.load(p['dreams_embeddings'],mmap_mode='r')
display(index.groupby('mode').agg(spectra=('spectrum_id','size'),min_precursor=('precursor_mz','min'),max_precursor=('precursor_mz','max')))
assert emb.shape==(len(index),1024) and np.isfinite(emb).all()
print('PASS：四类输入与 DreaMS 嵌入均已冻结，可进入第30步。')
